# DNABERT-2 Heavy Training on Google Colab

This notebook prepares a larger ClinVar GRCh38 alternate-sequence dataset and fine-tunes DNABERT-2 on a Colab GPU.

Research/education only. This is not a medical diagnosis tool.

## 1. GPU Check

In [1]:
import subprocess
import torch

print("nvidia-smi output:")
subprocess.run(["nvidia-smi"], check=False)

cuda_available = torch.cuda.is_available()
print("torch CUDA available:", cuda_available)

if not cuda_available:
    raise RuntimeError("No GPU is available. In Colab, go to Runtime > Change runtime type > GPU, then rerun.")

print("GPU name:", torch.cuda.get_device_name(0))

nvidia-smi output:
torch CUDA available: True
GPU name: Tesla T4


## 2. Clone GitHub Repo

In [2]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/faisalAI27/Variant-Risk-Explainer.git"
PLACEHOLDER = "PASTE_YOUR_GITHUB_REPO_URL_HERE"

if REPO_URL == PLACEHOLDER or not REPO_URL.strip():
    raise ValueError("Paste your GitHub repository URL into REPO_URL before running this cell.")

repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
repo_path = Path("/content") / repo_name

if not repo_path.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)
else:
    print(f"Repository already exists: {repo_path}")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)

os.chdir(repo_path)
print("Project root:", Path.cwd())

Project root: /content/Variant-Risk-Explainer


## 3. Install Dependencies

In [3]:
!python -m pip install --upgrade pip
!pip install -q torch transformers datasets accelerate scikit-learn pandas numpy safetensors tqdm biopython requests joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


## 5. Prepare Larger ClinVar Dataset

Optional: only run this 20k cell after 10k preparation and training work correctly.

In [4]:
RUN_20K = True

if RUN_20K:
    !python training/prepare_larger_clinvar_dataset.py --target_total 20000
else:
    print("20k preparation skipped. Set RUN_20K = True when ready.")

Prepare larger ClinVar dataset
Target total rows: 20,000
Output reference directory: /content/Variant-Risk-Explainer/training/csv_files_20k
Output alternate directory: /content/Variant-Risk-Explainer/training/csv_files_20k_alt
Flank size: 512

Existing source candidate: /content/Variant-Risk-Explainer/training/csv_files
  rows before filtering: 997
  rows after filtering: 906
Existing source candidate: /content/Variant-Risk-Explainer/training/csv_files_alt
  rows before filtering: 997
  rows after filtering: 906
Existing data is available but smaller than target_total. Parsing ClinVar VCF to build a larger candidate set.
Parsing ClinVar VCF: /content/Variant-Risk-Explainer/data/raw/clinvar.vcf.gz
Parsing VCF records: 4435014it [05:28, 13503.34it/s]
FILTERING SUMMARY
Selected source: /content/Variant-Risk-Explainer/data/raw/clinvar.vcf.gz
Total variants before filtering: 4,434,969
Total variants after filtering: 1,693,561
Class distribution after filtering:
label
0    1357917
1     3356

## 6. Check Dataset

In [5]:
!python training/check_large_dataset.py

Large ClinVar dataset check
Project root: /content/Variant-Risk-Explainer

20K ALTERNATE-SEQUENCE DATASET
Dataset directory: /content/Variant-Risk-Explainer/training/csv_files_20k_alt
--------------------------------------------------------------------------------
TRAIN SPLIT
--------------------------------------------------------------------------------
File path: /content/Variant-Risk-Explainer/training/csv_files_20k_alt/train_with_alt_sequences.csv
Rows: 14,000
Columns: ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'variant_type', 'gene_symbol', 'GENEINFO', 'CLNSIG', 'CLNHGVS', 'CLNVC', 'label', 'label_name', 'variant_id', 'sequence', 'ref_sequence', 'alt_sequence', 'ref_center', 'alt_center']
Label distribution:
label
0    7000
1    7000
Missing sequence count: 0
Sequence length min/mean/max: 1025 / 1025.28 / 1075
CLNSIG suspicious rows: 0
First 3 examples:
             variant_id REF ALT  label  sequence_length
  GRCh38-5-14287189-T-C   T   C      0             1025
  GRCh38-5-90721061-T-

## 7. Audit Alternate Sequence Encoding

Expected conclusion: `The sequence appears to contain alternate alleles.`

In [6]:
!python training/audit_variant_sequence_encoding.py

Variant sequence encoding audit
Selected data directory: /content/Variant-Risk-Explainer/training/csv_files_20k_alt
Auditing alternate dataset: True
Expected variant start index: 512

TRAIN SPLIT
Path: /content/Variant-Risk-Explainer/training/csv_files_20k_alt/train_with_alt_sequences.csv
Rows seen: 14,000
Total rows checked: 14,000
SNV rows checked: 10,606
Indel rows checked: 3,394
Reference allele matches: 0
Alternate allele matches: 14,000
Mismatches: 0
Skipped rows:
  missing sequence/REF/ALT: 0
  sequence too short for center check: 0
  multiple ALT alleles: 0
  symbolic ALT allele: 0
  non-ACGTN allele: 0
  not SNV or small indel: 0

VAL SPLIT
Path: /content/Variant-Risk-Explainer/training/csv_files_20k_alt/val_with_alt_sequences.csv
Rows seen: 3,000
Total rows checked: 3,000
SNV rows checked: 2,202
Indel rows checked: 798
Reference allele matches: 0
Alternate allele matches: 3,000
Mismatches: 0
Skipped rows:
  missing sequence/REF/ALT: 0
  sequence too short for center check: 0


## 8. GPU Training

In [7]:
%%bash
python training/train_local_dnabert2.py \
  --sample_size 0 \
  --epochs 5 \
  --batch_size 4 \
  --grad_accum_steps 4 \
  --learning_rate 2e-5 \
  --freeze_encoder true \
  --unfreeze_last_n_layers 4 \
  --use_class_weights true \
  --center_crop true \
  --tune_threshold true \
  --eval_accumulation_steps 8 \
  --save_eval_each_epoch false \
  --eval_subset_size 0

DNABERT-2 ClinVar training
Selected dataset directory: /content/Variant-Risk-Explainer/training/csv_files_20k_alt
Selected dataset name: 20k alternate-sequence dataset
Using 20k alt-sequence dataset: True
Using 10k alt-sequence dataset: False
Using large alt-sequence dataset: True
Selected train CSV path: /content/Variant-Risk-Explainer/training/csv_files_20k_alt/train_with_alt_sequences.csv
Selected validation CSV path: /content/Variant-Risk-Explainer/training/csv_files_20k_alt/val_with_alt_sequences.csv
Selected test CSV path: /content/Variant-Risk-Explainer/training/csv_files_20k_alt/test_with_alt_sequences.csv
Selected sequence column: sequence
Using alt-sequence dataset: True
Selected device: cuda
CUDA training enabled. fp16 will be enabled in TrainingArguments.
Output path: /content/Variant-Risk-Explainer/training/outputs/dnabert2_clinvar

train CSV: /content/Variant-Risk-Explainer/training/csv_files_20k_alt/train_with_alt_sequences.csv
train rows before filtering: 14,000
train r

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNABERT-2-117M:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNABERT-2-117M:
- flash_attn_triton.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNABERT-2-117M:
- bert_padding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNABERT-2-117M:
- bert_layers.py
- flash_attn_triton.py
- bert_padding.py

## 9. Full Evaluation

In [8]:
!python training/evaluate_saved_model.py --tune_threshold true

Memory-safe saved model evaluation
Saved model directory: /content/Variant-Risk-Explainer/training/outputs/dnabert2_clinvar/final_model
Selected dataset directory: /content/Variant-Risk-Explainer/training/csv_files_20k_alt
Selected device: cuda
Tune threshold on full validation set: True
Threshold search range: 0.1000 to 0.9000 by 0.0100
Using manual small-batch evaluation. No retraining. No HuggingFace Trainer evaluation.

Loading tokenizer and model.
CUDA detected. Trying to load saved model directly first.
Direct saved-model load failed.
Direct load error: /content/Variant-Risk-Explainer/training/outputs/dnabert2_clinvar/final_model does not appear to have a file named bert_layers.py. Checkout 'https://huggingface.co//content/Variant-Risk-Explainer/training/outputs/dnabert2_clinvar/final_model/tree/main' for available files.
Falling back to local no-Triton DNABERT-2 code.
Using local Mac-safe DNABERT-2 code: /content/Variant-Risk-Explainer/training/local_dnabert2_patch
Triton/flash 

## 10. Save Outputs to Google Drive

In [9]:
from pathlib import Path
import shutil

source_dir = Path("training/outputs/dnabert2_clinvar")
drive_output = Path("/content/drive/MyDrive/variant-risk-explainer")

if not drive_output.exists():
    print("Google Drive output folder is not available. Run the Drive mount cell first.")
else:
    drive_output.mkdir(parents=True, exist_ok=True)
    items_to_copy = [
        source_dir / "final_model",
        source_dir / "metrics.json",
        source_dir / "full_eval_metrics.json",
        source_dir / "final_dnabert2_clinvar_model.zip",
    ]

    for item in items_to_copy:
        if not item.exists():
            print(f"Skipping missing item: {item}")
            continue

        destination = drive_output / item.name
        if item.is_dir():
            if destination.exists():
                shutil.rmtree(destination)
            shutil.copytree(item, destination)
        else:
            shutil.copy2(item, destination)
        print(f"Copied {item} -> {destination}")

Google Drive output folder is not available. Run the Drive mount cell first.


## 11. Download Model Artifacts

In [10]:
from pathlib import Path
from google.colab import files

output_dir = Path("training/outputs/dnabert2_clinvar")
zip_path = output_dir / "final_dnabert2_clinvar_model.zip"

if not zip_path.exists():
    !cd training/outputs/dnabert2_clinvar && zip -r final_dnabert2_clinvar_model.zip final_model metrics.json full_eval_metrics.json

files.download(str(zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>